
# Milestone 2 — Schema-Aware Validation and Candidate Reranking

Building on the LoRA fine-tuned adapter from Milestone 1, this notebook introduces
schema validity enforcement and execution-guided candidate reranking as structured
output validation mechanisms.

## 1. Install Dependencies

Install and pin the same library versions used in Milestone 1 to ensure
compatibility with the saved adapter.

In [ ]:
!pip uninstall -y trl transformers peft accelerate
!pip install -q transformers==4.47.1 peft==0.13.2 accelerate==1.2.1 datasets sqlparse sentencepiece

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 122.0 MB/s eta 0:00:00


## 2. Verify Library Versions

In [ ]:
import transformers, peft, torch
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

transformers: 4.47.1
peft: 0.13.2
torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## 3. Mount Google Drive and Define All Paths

All inputs (adapter, preprocessed data) are loaded from Drive.
All outputs (CSVs, results) are saved back to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, re, sqlite3
import torch
import pandas as pd

drive.mount('/content/drive')

DRIVE_ROOT      = Path("/content/drive/MyDrive/CS_512_Project")
MILESTONE1_DIR  = DRIVE_ROOT / "milestone1_lora"
MILESTONE2_DIR  = DRIVE_ROOT / "milestone2_schema_val"
PROCESSED_DIR   = DRIVE_ROOT / "processed_data"
ADAPTER_DIR     = MILESTONE1_DIR / "final_adapter"
CSV_DIR         = MILESTONE2_DIR / "csv_outputs"
SPIDER_ROOT     = Path("/content/drive/MyDrive/CS_512_Project/spider/spider_data")

for p in [MILESTONE2_DIR, CSV_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive root:     ", DRIVE_ROOT)
print("Adapter dir:    ", ADAPTER_DIR)
print("Milestone 2 dir:", MILESTONE2_DIR)
print("Adapter exists: ", ADAPTER_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root:      /content/drive/MyDrive/CS_512_Project
Adapter dir:     /content/drive/MyDrive/CS_512_Project/milestone1_lora/final_adapter
Milestone 2 dir: /content/drive/MyDrive/CS_512_Project/milestone2_schema_val
Adapter exists:  True


## 4. Load Preprocessed Spider Data from Drive

Load the same processed_train.json and processed_dev.json saved by the
Baseline notebook. No preprocessing is repeated here.

In [ ]:
with open(PROCESSED_DIR / "processed_train.json", "r", encoding="utf-8") as f:
    processed_train = json.load(f)

with open(PROCESSED_DIR / "processed_dev.json", "r", encoding="utf-8") as f:
    processed_dev = json.load(f)

print("Loaded processed_train:", len(processed_train))
print("Loaded processed_dev:  ", len(processed_dev))
print("Sample keys:", list(processed_train[0].keys()))

Loaded processed_train: 2500
Loaded processed_dev:   250
Sample keys: ['db_id', 'question', 'prompt', 'target_sql', 'schema_text', 'sqlite_path', 'text']


## 5. Fix Stale SQLite Paths

The Baseline notebook saved absolute paths rooted at /content/text2sql_project/
which does not exist in this session. Remap all paths to the current Drive location
and verify at least one file resolves correctly.

In [ ]:
OLD_PREFIX = "/content/text2sql_project/spider_data"
NEW_PREFIX = str(SPIDER_ROOT)

def fix_sqlite_path(example):
    if example["sqlite_path"].startswith(OLD_PREFIX):
        example["sqlite_path"] = example["sqlite_path"].replace(OLD_PREFIX, NEW_PREFIX, 1)
    return example

processed_train = [fix_sqlite_path(ex) for ex in processed_train]
processed_dev   = [fix_sqlite_path(ex) for ex in processed_dev]

sample_path = processed_dev[0]["sqlite_path"]
print("Remapped path:", sample_path)
print("File exists:  ", os.path.exists(sample_path))
assert os.path.exists(sample_path), "SQLite path is wrong — check NEW_PREFIX"

Remapped path: /content/drive/MyDrive/CS_512_Project/spider/spider_data/database/concert_singer/concert_singer.sqlite
File exists:   True


## 6. Load Spider Tables for Schema Map

Build a lookup dictionary from db_id to full schema metadata. This is the
foundation for the schema validity checker — we need to know exactly which
tables and columns exist in each database.

In [ ]:
tables_path = SPIDER_ROOT / "tables.json"
with open(tables_path, "r", encoding="utf-8") as f:
    tables = json.load(f)

schema_map = {}
for db in tables:
    schema_map[db["db_id"]] = {
        "db_id":                  db["db_id"],
        "table_names_original":   db["table_names_original"],
        "column_names_original":  db["column_names_original"],
        "column_types":           db["column_types"],
        "primary_keys":           db["primary_keys"],
        "foreign_keys":           db["foreign_keys"],
    }

print("Total schemas loaded:", len(schema_map))
print("Sample db_ids:", list(schema_map.keys())[:5])

# Build a fast lookup: db_id → set of table names (lowercase)
#                      db_id → set of column names (lowercase)
table_name_sets  = {}
column_name_sets = {}

for db_id, schema in schema_map.items():
    table_name_sets[db_id]  = {t.lower() for t in schema["table_names_original"]}
    column_name_sets[db_id] = {
        col.lower()
        for _, col in schema["column_names_original"]
        if col != "*"
    }

print("Schema lookup sets built.")
print("Sample tables for 'concert_singer':", table_name_sets.get("concert_singer"))

NameError: name 'SPIDER_ROOT' is not defined

In [ ]:
def build_focused_schema(question: str, schema_text: str, db_id: str):
    """
    Keep only tables whose name or columns appear as keywords in the question.
    Falls back to full schema if no matches found.
    """
    question_words = set(question.lower().split())
    tables  = schema_map[db_id]["table_names_original"]
    columns = schema_map[db_id]["column_names_original"]

    relevant_table_indices = set()

    # Match table names
    for idx, table in enumerate(tables):
        if table.lower() in question_words:
            relevant_table_indices.add(idx)

    # Match column names
    for table_idx, col in columns:
        if table_idx == -1:
            continue
        if col.lower() in question_words:
            relevant_table_indices.add(table_idx)

    # Fall back to full schema if nothing matched
    if not relevant_table_indices:
        return schema_text

    # Rebuild schema text with only relevant tables and their columns
    table_to_cols = {i: [] for i in range(len(tables))}
    for col_idx, (table_idx, col_name) in enumerate(columns):
        if table_idx == -1:
            continue
        table_to_cols[table_idx].append(col_name)

    lines = ["Tables:"]
    for idx in sorted(relevant_table_indices):
        cols = table_to_cols[idx]
        lines.append(f"- {tables[idx]}({', '.join(cols)})")

    # Always include foreign keys — needed for JOIN correctness
    foreign_keys = schema_map[db_id].get("foreign_keys", [])
    if foreign_keys:
        lines.append("\nForeign Keys:")
        col_names_original = schema_map[db_id]["column_names_original"]
        table_names_original = schema_map[db_id]["table_names_original"]
        for src_idx, tgt_idx in foreign_keys:
            src_t, src_c = col_names_original[src_idx]
            tgt_t, tgt_c = col_names_original[tgt_idx]
            if src_t in relevant_table_indices or tgt_t in relevant_table_indices:
                lines.append(
                    f"- {table_names_original[src_t]}.{src_c} -> "
                    f"{table_names_original[tgt_t]}.{tgt_c}"
                )

    return "\n".join(lines)

## 7. Load Tokenizer and Both Models

Load the pure base model for comparison and the LoRA fine-tuned model from
the Milestone 1 adapter. Two completely separate model instances are loaded
to ensure a fair, independent comparison.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gc

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

# Model 1: Pure base — no adapter
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
base_model.eval()
print("Base model loaded.")

# Model 2: Base + LoRA adapter from Milestone 1
lora_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
tuned_model = PeftModel.from_pretrained(lora_base, str(ADAPTER_DIR))
tuned_model.eval()
print("LoRA tuned model loaded.")

# Confirm they are separate objects
print("Same object?", base_model is tuned_model)

Tokenizer loaded.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LoRA tuned model loaded.
Same object? False


## 8. Define SQL Generation Helper

A single generation function used by all models throughout this notebook.
Uses the chat template to match the training format from Milestone 1.

In [ ]:
def generate_sql(prompt, model, gen_tokenizer, max_new_tokens=256,
                 do_sample=False, temperature=1.0):
    messages = [
        {"role": "system",
         "content": "You are a text-to-SQL assistant. Return only valid SQLite SQL."},
        {"role": "user", "content": prompt},
    ]
    inputs = gen_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            pad_token_id=gen_tokenizer.eos_token_id,
            eos_token_id=gen_tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_ids = outputs[0, prompt_len:]
    return gen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def clean_sql(text: str) -> str:
    text = text.strip()
    if text.lower().startswith("sql"):
        text = text[3:].strip(": \n")
    if "```" in text:
        parts = text.split("```")
        text = parts[1] if len(parts) >= 2 else parts[0]
        text = re.sub(r"^sql\s*", "", text.strip(), flags=re.IGNORECASE)
    text = text.split("\n\n")[0].strip()
    text = text.split(";--")[0].strip()
    return text

## 9. Schema Validity Checker

This is the core new component of Milestone 2. Given a predicted SQL string
and a database schema, it checks whether all referenced table and column names
actually exist in that schema. This catches a large class of errors that
execution checking alone misses.

In [ ]:
import sqlparse
from sqlparse.sql import IdentifierList, Identifier, Where
from sqlparse.tokens import Keyword, DML

def extract_identifiers_from_sql(sql_text: str):
    """
    Extract all word tokens from a SQL string that could be table or column names.
    Uses a simple token scan — catches most cases without a full SQL parser.
    """
    # Remove string literals to avoid matching values as identifiers
    sql_clean = re.sub(r"'[^']*'", "", sql_text)
    sql_clean = re.sub(r'"[^"]*"', "", sql_clean)

    # Extract all word-like tokens (identifiers)
    tokens = re.findall(r'\b([a-zA-Z_][a-zA-Z0-9_]*)\b', sql_clean)

    # Filter out SQL keywords
    sql_keywords = {
        "select", "from", "where", "join", "on", "group", "by", "order",
        "having", "limit", "offset", "distinct", "count", "sum", "avg",
        "min", "max", "as", "and", "or", "not", "in", "like", "between",
        "is", "null", "case", "when", "then", "else", "end", "inner",
        "outer", "left", "right", "cross", "union", "all", "exists",
        "insert", "update", "delete", "create", "drop", "asc", "desc",
        "true", "false", "upper", "lower", "length", "substr", "trim"
    }
    return {t.lower() for t in tokens if t.lower() not in sql_keywords}


def build_column_table_pairs(db_id: str):
    """
    Returns a set of valid (table, column) pairs for a database,
    lowercased. Used to catch wrong-alias errors like T2.Make where
    Make belongs to T1.
    """
    schema = schema_map.get(db_id, {})
    tables = schema.get("table_names_original", [])
    columns = schema.get("column_names_original", [])

    valid_pairs = set()
    for table_idx, col_name in columns:
        if table_idx == -1:
            continue
        table_name = tables[table_idx].lower()
        valid_pairs.add(f"{table_name}.{col_name.lower()}")
    return valid_pairs

# Pre-build for all databases
column_table_pairs = {db_id: build_column_table_pairs(db_id)
                      for db_id in schema_map}


def schema_validity_check(sql_text: str, db_id: str):
    """
    Check whether all identifiers in the SQL exist in the database schema.

    Returns:
        is_valid (bool): True if no unknown identifiers found
        violations (list): list of identifiers not found in schema
        score (float): 0.0 to 1.0, fraction of identifiers that are valid
    """
    if not sql_text or not sql_text.strip():
        return False, ["empty SQL"], 0.0

    identifiers = extract_identifiers_from_sql(sql_text)
    if not identifiers:
        return True, [], 1.0

    valid_names = table_name_sets.get(db_id, set()) | column_name_sets.get(db_id, set())

    violations = [ident for ident in identifiers if ident not in valid_names]
    score = 1.0 - (len(violations) / len(identifiers))

    is_valid = len(violations) == 0
    return is_valid, violations, round(score, 4)


# Test on a sample example
sample = processed_dev[0]
test_sql = "SELECT name FROM singer WHERE age > 20"
is_valid, violations, score = schema_validity_check(test_sql, sample["db_id"])
print(f"DB: {sample['db_id']}")
print(f"SQL: {test_sql}")
print(f"Valid: {is_valid} | Score: {score} | Violations: {violations}")

# Test with a bad SQL
bad_sql = "SELECT fake_column FROM nonexistent_table WHERE x = 1"
is_valid2, violations2, score2 = schema_validity_check(bad_sql, sample["db_id"])
print(f"\nBad SQL: {bad_sql}")
print(f"Valid: {is_valid2} | Score: {score2} | Violations: {violations2}")

DB: concert_singer
SQL: SELECT name FROM singer WHERE age > 20
Valid: True | Score: 1.0 | Violations: []

Bad SQL: SELECT fake_column FROM nonexistent_table WHERE x = 1
Valid: False | Score: 0.0 | Violations: ['fake_column', 'x', 'nonexistent_table']


## 10. Execution Helper

Runs a SQL string against the actual SQLite database and returns whether
it executed successfully and what result set it produced.

In [ ]:
def execute_sql(db_path: str, sql: str):
    try:
        conn = sqlite3.connect(db_path)
        cur  = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return True, rows, None
    except Exception as e:
        return False, None, str(e)


def normalize_sql(sql_text: str) -> str:
    sql = str(sql_text).strip().lower().replace(";", "")
    sql = re.sub(r"\s+",    " ",  sql)
    sql = re.sub(r"\s*,\s*", ", ", sql)
    sql = re.sub(r"\s*\(\s*", "(", sql)
    sql = re.sub(r"\s*\)\s*", ")", sql)
    sql = re.sub(r"\s*=\s*", " = ", sql)
    return re.sub(r"\s+", " ", sql).strip()

## 11. Candidate Generator and Reranker

Generate N diverse SQL candidates for each question using the tuned model,
then rank them using three signals in priority order:
  1. Schema validity score (are all table/column names real?)
  2. Executability (does the SQL run without error?)
  3. Candidate index (prefer greedy/earlier candidates as tiebreaker)

In [ ]:
def generate_candidates(prompt: str, model, gen_tokenizer,
                         n_candidates: int = 5):
    """
    Generate n_candidates SQL strings for a given prompt.
    First candidate is greedy (deterministic best guess).
    Remaining candidates use sampling with increasing temperature
    for diversity.
    """
    candidates = []

    # Candidate 0: greedy decode — most confident prediction
    sql = clean_sql(generate_sql(prompt, model, gen_tokenizer,
                                  do_sample=False))
    candidates.append(sql)

    # Candidates 1–N: sampled with increasing temperature for diversity
    temperatures = [0.3, 0.5, 0.7, 1.0]
    for temp in temperatures[:n_candidates - 1]:
        sql = clean_sql(generate_sql(prompt, model, gen_tokenizer,
                                      do_sample=True, temperature=temp))
        candidates.append(sql)

    return candidates


def rerank_candidates(candidates: list, db_id: str, db_path: str):
    """
    Score and rank candidates. Returns the best SQL and a detailed
    score breakdown for every candidate.

    Ranking priority:
      1. Schema validity score (higher = more real identifiers)
      2. Executability (executes without error = True > False)
      3. Original order (greedy first = preferred tiebreaker)
    """
    scored = []

    for rank, sql in enumerate(candidates):
        is_valid, violations, schema_score = schema_validity_check(sql, db_id)
        exec_ok, rows, exec_err = execute_sql(db_path, sql)

        # NEW: row count as a proxy signal — more specific queries
        # (fewer rows) are usually more correct for Spider questions
        row_count = len(rows) if exec_ok and rows else 0


        scored.append({
            "sql":          sql,
            "schema_score": schema_score,
            "is_valid":     is_valid,
            "exec_ok":      exec_ok,
            "exec_err":     exec_err,
            "violations":   violations,
            "row_count":    row_count,
            "original_rank": rank,
        })

    # Sort: primary = schema_score descending,
    #       secondary = exec_ok descending (True > False),
    #       tertiary = original_rank ascending (greedy preferred)
    scored.sort(key=lambda x: (
        -x["schema_score"],
        -int(x["exec_ok"]),
         x["original_rank"],
    ))

    best = scored[0]["sql"]
    return best, scored

from collections import Counter

def self_consistency_pick(candidates: list) -> str:
    """
    Pick the SQL that appears most frequently among candidates
    after normalization. Ties broken by original order.
    """
    if not candidates:
        return ""
    normalized = [normalize_sql(c) for c in candidates]
    most_common_norm = Counter(normalized).most_common(1)[0][0]
    for c in candidates:
        if normalize_sql(c) == most_common_norm:
            return c
    return candidates[0]


# Quick sanity test
sample_ex = processed_dev[0]
print("Question:", sample_ex["question"])
print("Gold SQL:", sample_ex["target_sql"])
print("\nGenerating candidates...")

candidates = generate_candidates(
    sample_ex["prompt"], tuned_model, tokenizer, n_candidates=3
)
best_sql, scores = rerank_candidates(
    candidates, sample_ex["db_id"], sample_ex["sqlite_path"]
)

print("\nAll candidates:")
for s in scores:
    print(f"  Rank {s['original_rank']} | schema={s['schema_score']} "
          f"| exec={s['exec_ok']} | sql: {s['sql'][:80]}")
print("\nBest SQL selected:", best_sql)

Question: How many singers do we have?
Gold SQL: SELECT count(*) FROM singer

Generating candidates...

All candidates:
  Rank 0 | schema=1.0 | exec=True | sql: SELECT count(*) FROM singer
  Rank 1 | schema=1.0 | exec=True | sql: SELECT count(*) FROM singer
  Rank 2 | schema=1.0 | exec=True | sql: SELECT count(*) FROM singer

Best SQL selected: SELECT count(*) FROM singer


## 12. Full Milestone 2 Evaluation Loop

Evaluate all 200 dev examples across four systems:
  1. Base model (no fine-tuning, no reranking)
  2. Tuned model — single greedy output (Milestone 1 result)
  3. Tuned model + schema validity filter (new)
  4. Tuned model + full candidate reranking (new)

In [ ]:
import time

N_CANDIDATES = 5
DEV_LIMIT    = 200

validation_examples = processed_dev[:DEV_LIMIT]
full_rows    = []
loop_start   = time.time()

for i, ex in enumerate(validation_examples):
    t0 = time.time()
    print(f"Example {i:>3}/{DEV_LIMIT} | {ex['db_id']:<20}", end=" ", flush=True)

    focused_schema = build_focused_schema(
    ex["question"], ex["schema_text"], ex["db_id"]
    )
    # Rebuild prompt with focused schema
    from_marker = ex["prompt"].find("Schema:")
    to_marker   = ex["prompt"].find("Question:")
    if from_marker != -1 and to_marker != -1:
        prompt = (ex["prompt"][:from_marker] +
                  "Schema:\n" + focused_schema + "\n\n" +
                  ex["prompt"][to_marker:])
    else:
        prompt = ex["prompt"]   # fallback if markers not found
    gold_sql = ex["target_sql"]
    db_id    = ex["db_id"]
    db_path  = ex["sqlite_path"]

    # ── 1. Base model ─────────────────────────────────────────────────────────
    base_sql = clean_sql(generate_sql(prompt, base_model, tokenizer))
    print(f"base✓", end=" ", flush=True)

    # ── 2. Tuned model greedy ─────────────────────────────────────────────────
    tuned_sql = clean_sql(generate_sql(prompt, tuned_model, tokenizer))
    print(f"tuned✓", end=" ", flush=True)

    # ── 3. Candidates for schema filter + reranker ────────────────────────────
    candidates = generate_candidates(prompt, tuned_model, tokenizer, N_CANDIDATES)
    print(f"cands✓", end=" ", flush=True)

    # ── 4. Schema filter ──────────────────────────────────────────────────────
    schema_best = tuned_sql
    for cand in candidates:
        is_v, _, _ = schema_validity_check(cand, db_id)
        if is_v:
            schema_best = cand
            break

    # ── 5. Reranker ───────────────────────────────────────────────────────────
    reranked_sql, candidate_scores = rerank_candidates(candidates, db_id, db_path)
    print(f"rerank✓", end=" ", flush=True)

    # ── Evaluate ──────────────────────────────────────────────────────────────
    gold_ok,   gold_res,   _          = execute_sql(db_path, gold_sql)
    base_ok,   base_res,   base_err   = execute_sql(db_path, base_sql)
    tuned_ok,  tuned_res,  tuned_err  = execute_sql(db_path, tuned_sql)
    schema_ok, schema_res, schema_err = execute_sql(db_path, schema_best)
    rerank_ok, rerank_res, rerank_err = execute_sql(db_path, reranked_sql)

    gold_norm    = normalize_sql(gold_sql)
    base_exact   = normalize_sql(base_sql)    == gold_norm
    tuned_exact  = normalize_sql(tuned_sql)   == gold_norm
    schema_exact = normalize_sql(schema_best) == gold_norm
    rerank_exact = normalize_sql(reranked_sql)== gold_norm

    base_exec   = gold_ok and base_ok   and gold_res == base_res
    tuned_exec  = gold_ok and tuned_ok  and gold_res == tuned_res
    schema_exec = gold_ok and schema_ok and gold_res == schema_res
    rerank_exec = gold_ok and rerank_ok and gold_res == rerank_res

    _, _, base_schema_score   = schema_validity_check(base_sql,     db_id)
    _, _, tuned_schema_score  = schema_validity_check(tuned_sql,    db_id)
    _, _, schema_schema_score = schema_validity_check(schema_best,  db_id)
    _, _, rerank_schema_score = schema_validity_check(reranked_sql, db_id)

    consistency_sql = self_consistency_pick(candidates)
    consistency_ok, consistency_res, consistency_err = execute_sql(db_path, consistency_sql)
    consistency_exec  = gold_ok and consistency_ok and gold_res == consistency_res
    consistency_exact = normalize_sql(consistency_sql) == gold_norm
    _, _, consistency_schema_score = schema_validity_check(consistency_sql, db_id)

    full_rows.append({
        "example_id":           i,
        "db_id":                db_id,
        "question":             ex["question"],
        "gold_sql":             gold_sql,
        "base_sql":             base_sql,
        "tuned_sql":            tuned_sql,
        "schema_best_sql":      schema_best,
        "reranked_sql":         reranked_sql,
        "sqlite_path":          db_path,
        "base_exact":           base_exact,
        "tuned_exact":          tuned_exact,
        "schema_exact":         schema_exact,
        "rerank_exact":         rerank_exact,
        "base_exec":            base_exec,
        "tuned_exec":           tuned_exec,
        "schema_exec":          schema_exec,
        "rerank_exec":          rerank_exec,
        "base_schema_score":    base_schema_score,
        "tuned_schema_score":   tuned_schema_score,
        "schema_schema_score":  schema_schema_score,
        "rerank_schema_score":  rerank_schema_score,
        "base_err":             base_err,
        "tuned_err":            tuned_err,
        "schema_err":           schema_err,
        "rerank_err":           rerank_err,
        "consistency_sql":          consistency_sql,
        "consistency_exec":         consistency_exec,
        "consistency_exact":        consistency_exact,
        "consistency_schema_score": consistency_schema_score,
        "consistency_err":          consistency_err,
    })

    t1 = time.time()
    print(f"| {t1-t0:.1f}s")

    # Checkpoint + progress every 20 examples
    if (i + 1) % 20 == 0:
        elapsed = time.time() - loop_start
        avg     = elapsed / (i + 1)
        eta     = avg * (DEV_LIMIT - (i + 1))
        print(f"\n{'─'*60}")
        print(f"Progress: {i+1}/{DEV_LIMIT} | "
              f"Elapsed: {elapsed/60:.1f}min | "
              f"ETA: {eta/60:.1f}min")
        checkpoint_df = pd.DataFrame(full_rows)
        checkpoint_df.to_csv(CSV_DIR / "milestone2_eval_checkpoint.csv", index=False)
        print(f"Checkpoint saved. ({len(full_rows)} rows)")
        print(f"{'─'*60}\n")

eval_df = pd.DataFrame(full_rows)
print(f"\nDone. Total: {len(eval_df)} examples in "
      f"{(time.time()-loop_start)/60:.1f} minutes")

Example   0/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 7.0s
Example   1/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 6.8s
Example   2/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 11.9s
Example   3/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 12.3s
Example   4/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 20.6s
Example   5/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 22.2s
Example   6/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 21.9s
Example   7/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 18.7s
Example   8/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 16.5s
Example   9/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 12.1s
Example  10/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 13.0s
Example  11/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 11.6s
Example  12/200 | concert_singer       base✓ tuned✓ cands✓ rerank✓ | 37.8s
Example  13/200 | concert_s

## 13. Milestone 2 Results — Ablation Table

Print the headline numbers comparing all four systems. This is the
primary results table for the milestone report.

In [ ]:
total = len(eval_df)

print("=" * 60)
print(f"{'System':<30} {'Exec Acc':>10} {'Exact Match':>12}")
print("=" * 60)

systems = [
    ("Base model",                "base_exec",   "base_exact"),
    ("+ LoRA fine-tuning (M1)",   "tuned_exec",  "tuned_exact"),
    ("+ Schema validity filter",  "schema_exec", "schema_exact"),
    ("+ Full reranking (M2)",     "rerank_exec", "rerank_exact"),
    ("+ Self-consistency",        "consistency_exec", "consistency_exact"),
]

for name, exec_col, exact_col in systems:
    exec_acc  = eval_df[exec_col].sum()  / total
    exact_acc = eval_df[exact_col].sum() / total
    print(f"{name:<30} {exec_acc:>9.1%} {exact_acc:>11.1%}")

print("=" * 60)
print(f"\nTotal dev examples: {total}")

print("\nMean schema validity scores:")
print(f"  Base model:    {eval_df['base_schema_score'].mean():.4f}")
print(f"  Tuned model:   {eval_df['tuned_schema_score'].mean():.4f}")
print(f"  Schema filter: {eval_df['schema_schema_score'].mean():.4f}")
print(f"  Reranked:      {eval_df['rerank_schema_score'].mean():.4f}")

System                           Exec Acc  Exact Match
Base model                         46.5%        3.5%
+ LoRA fine-tuning (M1)            49.5%       37.5%
+ Schema validity filter           50.0%       38.5%
+ Full reranking (M2)              52.0%       38.0%
+ Self-consistency                 49.5%       37.5%

Total dev examples: 200

Mean schema validity scores:
  Base model:    0.8258
  Tuned model:   0.8394
  Schema filter: 0.8472
  Reranked:      0.8588


## 14. Outcome Analysis

Break down where each system wins and loses relative to the others.
This identifies which types of examples benefit most from schema
validation and reranking.

In [ ]:
# Where reranking beats tuned-greedy
rerank_beats_tuned = eval_df[eval_df["rerank_exec"] & ~eval_df["tuned_exec"]]
tuned_beats_rerank = eval_df[eval_df["tuned_exec"]  & ~eval_df["rerank_exec"]]
both_correct       = eval_df[eval_df["rerank_exec"] & eval_df["tuned_exec"]]
both_wrong         = eval_df[~eval_df["rerank_exec"] & ~eval_df["tuned_exec"]]

print("Reranking vs Tuned-Greedy:")
print(f"  Reranking better (exec): {len(rerank_beats_tuned)}")
print(f"  Tuned better (exec):     {len(tuned_beats_rerank)}")
print(f"  Both correct:            {len(both_correct)}")
print(f"  Both wrong:              {len(both_wrong)}")

# Schema validity improvement
print("\nSchema validity — fraction of examples with fully valid SQL:")
print(f"  Base:      {(eval_df['base_schema_score']   == 1.0).sum()} / {total}")
print(f"  Tuned:     {(eval_df['tuned_schema_score']  == 1.0).sum()} / {total}")
print(f"  Reranked:  {(eval_df['rerank_schema_score'] == 1.0).sum()} / {total}")

# Error type analysis on remaining failures
print("\nTop errors in reranked failures:")
failed = eval_df[~eval_df["rerank_exec"]]["rerank_err"].dropna()
error_counts = failed.value_counts().head(10)
print(error_counts)

Reranking vs Tuned-Greedy:
  Reranking better (exec): 7
  Tuned better (exec):     2
  Both correct:            97
  Both wrong:              94

Schema validity — fraction of examples with fully valid SQL:
  Base:      74 / 200
  Tuned:     104 / 200
  Reranked:  109 / 200

Top errors in reranked failures:
rerank_err
no such column: model                   5
no such column: T1.stadium_name         2
no such column: T1.Pet                  2
no such column: T1.Pet_id               2
no such column: pet                     2
no such column: T2.Song_Name            2
no such column: CountryId               2
no such column: T2.MakeId               2
no such column: T1.Song_Release_Year    1
no such table: artist                   1
Name: count, dtype: int64


## 15. Save All Results to Drive

In [ ]:
output_csv = CSV_DIR / "milestone2_full_eval.csv"
eval_df.to_csv(output_csv, index=False)
print("Saved full evaluation to:", output_csv)

# Save a compact summary
summary = {
    "total_examples": total,
    "base_exec_acc":    round(eval_df["base_exec"].mean(),   4),
    "tuned_exec_acc":   round(eval_df["tuned_exec"].mean(),  4),
    "schema_exec_acc":  round(eval_df["schema_exec"].mean(), 4),
    "rerank_exec_acc":  round(eval_df["rerank_exec"].mean(), 4),
    "base_exact_acc":   round(eval_df["base_exact"].mean(),  4),
    "tuned_exact_acc":  round(eval_df["tuned_exact"].mean(), 4),
    "schema_exact_acc": round(eval_df["schema_exact"].mean(),4),
    "rerank_exact_acc": round(eval_df["rerank_exact"].mean(),4),
    "mean_base_schema_score":   round(eval_df["base_schema_score"].mean(),   4),
    "mean_tuned_schema_score":  round(eval_df["tuned_schema_score"].mean(),  4),
    "mean_rerank_schema_score": round(eval_df["rerank_schema_score"].mean(), 4),
}

summary_path = CSV_DIR / "milestone2_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to:", summary_path)
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Saved full evaluation to: /content/drive/MyDrive/CS_512_Project/milestone2_schema_val/csv_outputs/milestone2_full_eval.csv
Saved summary to: /content/drive/MyDrive/CS_512_Project/milestone2_schema_val/csv_outputs/milestone2_summary.json

Summary:
  total_examples: 200
  base_exec_acc: 0.465
  tuned_exec_acc: 0.495
  schema_exec_acc: 0.5
  rerank_exec_acc: 0.52
  base_exact_acc: 0.035
  tuned_exact_acc: 0.375
  schema_exact_acc: 0.385
  rerank_exact_acc: 0.38
  mean_base_schema_score: 0.8258
  mean_tuned_schema_score: 0.8394
  mean_rerank_schema_score: 0.8588
